# KernelMigrate: Can AI Keep Accelerated Code Alive?

**KernelMigrate** evaluates AI-driven GPU-kernel maintenance rather than greenfield kernel generation.

The core question is not merely:

> Can an AI write a kernel from scratch?

It is:

> Can an AI preserve a known-good kernel through backend ports, architecture changes, runtime upgrades, stale autotuning, and years of dependency drift?

[View the KernelMigrate repository on GitHub](https://github.com/AdamBoxy/KernelMigrate-Bench)

In [1]:
# Cell 1 — clone your repository
!git clone --depth 1 https://github.com/AdamBoxy/KernelMigrate-Bench.git

%cd /kaggle/working/KernelMigrate-Bench

Cloning into 'KernelMigrate-Bench'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 45 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 17.15 KiB | 8.57 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/kaggle/working/KernelMigrate-Bench


In [2]:
# Cell 2 — sanity-check the repository contents
!find . -maxdepth 3 -type f | sort

./docs/AGENT_PROMPT.md
./docs/HARDWARE_EXECUTOR.md
./.git/config
./.git/description
./.git/HEAD
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/update.sample
./.github/workflows/codeql.yml
./.github/workflows/python-app.yml
./.git/index
./.git/info/exclude
./.git/logs/HEAD
./.git/packed-refs
./.git/shallow
./kernelmigrate.py
./LICENSE
./NOTICE.md
./README.md
./reference_solutions/cuda_to_hip/solution.json
./reference_solutions/gfx942_to_gfx950/solution.json
./reference_solutions/rocm_n_to_n1/solution.json
./reference_solutions/stale_autotune/solution.json
./reference_solutions/three_year_neglect/solution.json
./results/

## Reproducibility

This notebook clones the public benchmark repository, runs its full test suite, prepares an intentionally stale maintenance workspace, and generates a scored HTML report.

> **Notebook setting:** Enable Internet access for the repository clone.  
> KernelMigrate v0.1 itself requires only Python's standard library and runs on Kaggle CPU.

In [3]:
# Cell 3 — run the benchmark's test suite
!python -m unittest discover -s tests -v

test_missing_candidate_does_not_crash (test_kernelmigrate.KernelMigrateTests.test_missing_candidate_does_not_crash) ... ok
test_reference_scores_100 (test_kernelmigrate.KernelMigrateTests.test_reference_scores_100) ... ok
test_semantic_failure_caps_task (test_kernelmigrate.KernelMigrateTests.test_semantic_failure_caps_task) ... ok
test_stale_tuning_is_slower (test_kernelmigrate.KernelMigrateTests.test_stale_tuning_is_slower) ... ok
test_starter_is_broken_but_partially_preserves_behavior (test_kernelmigrate.KernelMigrateTests.test_starter_is_broken_but_partially_preserves_behavior) ... ok
test_wrong_target_caps_task (test_kernelmigrate.KernelMigrateTests.test_wrong_target_caps_task) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.004s

OK


## Benchmark Tracks

KernelMigrate v0.1 contains five maintenance scenarios:

1. **CUDA → HIP** — porting across programming stacks.
2. **gfx942 → gfx950** — retargeting architecture-specific assumptions.
3. **ROCm N → N+1** — restoring compatibility after a runtime upgrade.
4. **Stale autotuning** — refreshing configurations inherited from older hardware.
5. **Three years of neglect** — a combined backend, runtime, architecture, and tuning migration.

In [4]:
# Cell 4 — show the five maintenance scenarios
!python kernelmigrate.py list

cuda_to_hip: Port an AXPY kernel from CUDA 12.4 on sm_90 to ROCm 7.0 on gfx942.
gfx942_to_gfx950: Retarget a tuned reduction kernel from gfx942 to gfx950.
rocm_n_to_n1: Restore a layer-normalization kernel after a ROCm 6.4 to 7.0 upgrade.
stale_autotune: Refresh a matrix-vector kernel configuration copied from gfx942 for gfx950.
three_year_neglect: Recover a fused bias+GELU kernel across backend, architecture, runtime, and tuning drift.


## Evaluation Workflow

Each task begins with a deliberately stale `solution.json` describing the legacy implementation.

An agent may update only the prepared candidate workspace. The evaluator then scores:

- semantic preservation of the old kernel contract;
- migration completeness;
- target-architecture compatibility;
- estimated performance retention;
- maintenance hygiene and validation evidence.

Critical semantic failures and incorrect architecture targets are score-capped.  

**For an actual agent run later, Cell 5 is the handoff point: give the agent permission to edit only `/kaggle/working/kernelmigrate_candidate`, then rerun Cells 6–7. That keeps the evaluator and reference solutions outside its writable scope.**

In [5]:
# Cell 5 — materialize the intentionally stale agent workspace
!rm -rf /kaggle/working/kernelmigrate_candidate
!python kernelmigrate.py prepare /kaggle/working/kernelmigrate_candidate

Prepared /kaggle/working/kernelmigrate_candidate


## Baseline: Untouched Legacy State

First, we grade the intentionally stale starting point.

This is not an agent failure, it establishes the benchmark's initial maintenance debt before any migration work occurs.

In [6]:
# Cell 6 — grade the untouched “three years of neglect” baseline
!python kernelmigrate.py grade \
    /kaggle/working/kernelmigrate_candidate \
    --output /kaggle/working/results/starter

kernelmigrate_candidate: 49.99/100


In [7]:
# Cell 7 — display the baseline report inside the notebook
from IPython.display import HTML, display

display(HTML(filename="/kaggle/working/results/starter/report.html"))

## Reference: Maintained Migration

The included reference solutions demonstrate a fully maintained state and provide a reproducible scoring target.

In a blinded future version, reference solutions and randomized verifier inputs would remain private while agents receive only the old repository, target environment, and migration brief.

In [8]:
# Cell 8 — grade the included maintained reference solutions
!python kernelmigrate.py grade \
    /kaggle/working/KernelMigrate-Bench/reference_solutions \
    --output /kaggle/working/results/reference

reference_solutions: 100/100


In [9]:
# Cell 9 — display the 100/100 reference report
display(HTML(filename="/kaggle/working/results/reference/report.html"))

## Attribution and Next Steps

The originating idea came from a question by **Paige Bailey**: can AI maintain and migrate kernels across changing hardware and software stacks, not merely generate new ones?

KernelMigrate v0.1 was designed and implemented by **BoxyML & Nyra**.

Repository: [github.com/AdamBoxy/KernelMigrate-Bench](https://github.com/AdamBoxy/KernelMigrate-Bench)

In [10]:
# Cell 10 — package reports for notebook output/download
!cd /kaggle/working && zip -r kernelmigrate_reports.zip results

print("Created: /kaggle/working/kernelmigrate_reports.zip")

  adding: results/ (stored 0%)
  adding: results/starter/ (stored 0%)
  adding: results/starter/report.html (deflated 76%)
  adding: results/starter/report.json (deflated 90%)
  adding: results/reference/ (stored 0%)
  adding: results/reference/report.html (deflated 78%)
  adding: results/reference/report.json (deflated 92%)
Created: /kaggle/working/kernelmigrate_reports.zip
